# Understanding Column Recipes
This notebook continues [`column-schema-and-the-bdf`](column-schema-and-the-bdf.ipynb) and focuses on one question: what does a recipe look like when PyProBE applies it?

Each example below removes a target column, asks `Table.get(...)` to reconstruct it, and plots the input columns above the resolved output.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import polars as pl

import pyprobe
from pyprobe.columns import BDF

warnings.filterwarnings(
    "ignore",
    category=DeprecationWarning,
)

plt.style.use("seaborn-v0_8-whitegrid")

In [ ]:
LABELS = {
    BDF.TEST_TIME_SECOND.name: "Test Time / h",
    BDF.CURRENT_AMPERE.name: "Current / A",
    BDF.CHARGING_CAPACITY_AH.name: "Charging Capacity / Ah",
    BDF.DISCHARGING_CAPACITY_AH.name: "Discharging Capacity / Ah",
    BDF.STEP_CHARGING_CAPACITY_AH.name: "Step Charging Capacity / Ah",
    BDF.STEP_DISCHARGING_CAPACITY_AH.name: "Step Discharging Capacity / Ah",
    BDF.NET_CAPACITY_AH.name: "Net Capacity / Ah",
}


def add_step_markers(ax, time_hours, step_count):
    if step_count is None:
        return
    for index in range(1, len(step_count)):
        if step_count[index] != step_count[index - 1]:
            ax.axvline(
                time_hours[index],
                color="0.75",
                linestyle="--",
                linewidth=1,
                zorder=0,
            )


def plot_recipe_application(
    source_df: pl.DataFrame,
    input_columns: list[str],
    plot_columns: list[str],
    target_column: BDF,
    title: str,
    recipe_caption: str,
):
    source_table = pyprobe.Table(
        lf=source_df.select(input_columns),
        metadata={"example": title},
    )
    resolved_target = source_table.get(target_column)

    select_columns = [BDF.TEST_TIME_SECOND.name, *plot_columns]
    if BDF.STEP_COUNT.name in source_df.columns:
        select_columns.insert(1, BDF.STEP_COUNT.name)
    plot_df = source_df.select(select_columns).with_columns(
        pl.Series(target_column.name, resolved_target)
    )

    time_hours = plot_df[BDF.TEST_TIME_SECOND.name].to_numpy() / 3600.0
    step_count = (
        plot_df[BDF.STEP_COUNT.name].to_numpy()
        if BDF.STEP_COUNT.name in plot_df.columns
        else None
    )

    fig, axes = plt.subplots(
        2,
        1,
        figsize=(10, 6),
        sharex=True,
        height_ratios=[1.15, 1.0],
        constrained_layout=True,
    )
    fig.suptitle(title, fontsize=15, fontweight="bold")

    source_colors = ["#1f77b4", "#ff7f0e", "#2ca02c"]
    for color, column in zip(source_colors, plot_columns):
        axes[0].plot(
            time_hours,
            plot_df[column].to_numpy(),
            marker="o",
            linewidth=2,
            markersize=5,
            label=LABELS[column],
            color=color,
        )
    add_step_markers(axes[0], time_hours, step_count)
    axes[0].set_ylabel("Available inputs")
    axes[0].legend(loc="upper left")
    axes[0].set_title(recipe_caption, loc="left", fontsize=11)

    axes[1].plot(
        time_hours,
        plot_df[target_column.name].to_numpy(),
        marker="o",
        linewidth=2.5,
        markersize=5,
        color="#d62728",
    )
    add_step_markers(axes[1], time_hours, step_count)
    axes[1].axhline(0.0, color="0.55", linewidth=1)
    axes[1].set_ylabel(LABELS[target_column.name])
    axes[1].set_xlabel("Test Time / h")
    axes[1].set_title("Resolved target", loc="left", fontsize=11)

    return fig

## 1. Direct recipe
Here the only available throughput columns are global charging and global discharging capacity. PyProBE resolves `Net Capacity / Ah` directly from those two traces.

In [ ]:
direct_df = pl.DataFrame(
    {
        BDF.TEST_TIME_SECOND.name: [0.0, 3600.0, 7200.0, 10800.0],
        BDF.CHARGING_CAPACITY_AH.name: [0.0, 1.0, 3.0, 3.0],
        BDF.DISCHARGING_CAPACITY_AH.name: [0.0, 0.0, 0.0, 2.0],
    }
)

In [ ]:
plot_recipe_application(
    source_df=direct_df,
    input_columns=[
        BDF.TEST_TIME_SECOND.name,
        BDF.CHARGING_CAPACITY_AH.name,
        BDF.DISCHARGING_CAPACITY_AH.name,
    ],
    plot_columns=[
        BDF.CHARGING_CAPACITY_AH.name,
        BDF.DISCHARGING_CAPACITY_AH.name,
    ],
    target_column=BDF.NET_CAPACITY_AH,
    title="Global net capacity from charging and discharging",
    recipe_caption="Recipe: charging - discharging -> net",
)
plt.show()

## 2. Step recipe with seam carry-over
This example uses the dedicated nonzero-seam fixture: one hourly charge step followed by one hourly discharge step. The dashed line marks the real step seam where the recipe adds back the missing trapezoidal carry-over.

In [ ]:
seam_df = pl.DataFrame(
    {
        BDF.TEST_TIME_SECOND.name: [0.0, 3600.0, 7200.0, 10800.0],
        BDF.STEP_COUNT.name: [0, 0, 1, 1],
        BDF.CURRENT_AMPERE.name: [2.0, 2.0, -1.0, -1.0],
        BDF.STEP_CHARGING_CAPACITY_AH.name: [0.0, 2.0, 0.0, 0.0],
        BDF.STEP_DISCHARGING_CAPACITY_AH.name: [0.0, 0.0, 0.0, 1.0],
    }
)

In [ ]:
plot_recipe_application(
    source_df=seam_df,
    input_columns=[
        BDF.TEST_TIME_SECOND.name,
        BDF.STEP_COUNT.name,
        BDF.CURRENT_AMPERE.name,
        BDF.STEP_CHARGING_CAPACITY_AH.name,
        BDF.STEP_DISCHARGING_CAPACITY_AH.name,
    ],
    plot_columns=[
        BDF.STEP_CHARGING_CAPACITY_AH.name,
        BDF.STEP_DISCHARGING_CAPACITY_AH.name,
        BDF.CURRENT_AMPERE.name,
    ],
    target_column=BDF.NET_CAPACITY_AH,
    title="Global net capacity from step columns",
    recipe_caption=("Recipe: (step charging - step discharging) + seam carry-over"),
)
plt.show()

## 3. Integration recipe
This final plot uses the same seam example as above, but removes the step capacity columns. Reconstructing `Net Capacity / Ah` from only current and elapsed time lands on the same resolved target, showing that the seam-aware step recipe and the integral agree.

In [ ]:
integration_df = seam_df.select(
    [
        BDF.TEST_TIME_SECOND.name,
        BDF.CURRENT_AMPERE.name,
    ]
)

In [ ]:
plot_recipe_application(
    source_df=integration_df,
    input_columns=[
        BDF.TEST_TIME_SECOND.name,
        BDF.CURRENT_AMPERE.name,
    ],
    plot_columns=[
        BDF.CURRENT_AMPERE.name,
    ],
    target_column=BDF.NET_CAPACITY_AH,
    title="Global net capacity from current integration",
    recipe_caption="Recipe: integrate current over test time",
)
plt.show()